In [4]:
!pip install sumy transformers nltk --quiet

# Импорты
import os
import pandas as pd
from tqdm import tqdm
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer
from sumy.nlp.stemmers import Stemmer
import nltk
import torch
from transformers import T5ForConditionalGeneration, AutoTokenizer

In [9]:
nltk.download('punkt')
nltk.download('punkt_tab')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Используемое устройство: {device}")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Используемое устройство: cpu


In [7]:
articles_path = '/content/10_articles'

texts = []
for file in sorted(os.listdir(articles_path)):
    if file.endswith(".txt"):
        with open(os.path.join(articles_path, file), 'r', encoding='utf-8') as f:
            text = f.read()
            name = file[:-4]
            texts.append((name, text))

df = pd.DataFrame(texts, columns=['name', 'text'])
print(f"Загружено {len(df)} статей")

Загружено 10 статей


In [10]:
def summarize_with_sumy(text, num_sentences=3):
    parser = PlaintextParser.from_string(text, Tokenizer("russian"))
    stemmer = Stemmer("russian")
    summarizer = LexRankSummarizer(stemmer)
    summary = summarizer(parser.document, num_sentences)
    return " ".join(str(sentence) for sentence in summary)

print("\n--- АННОТАЦИИ С SUMY ---\n")
sumy_annotations = []
for i, row in df.iterrows():
    summary = summarize_with_sumy(row['text'])
    sumy_annotations.append(summary)
    print(f"Статья {row['name']}:\n{summary}\n")


--- АННОТАЦИИ С SUMY ---

Статья Adrova_IMS_2013_rus:
Время хеширования этого алгоритма достигает 0,046-0,047 мс. Алгоритм хеширования BCRYPT использует соль для защиты от радужных таблиц. Исследование алгоритмов хеширования в зависимости от числа используемых раундов Второй подход к борьбе с полным перебором паролей – увеличение числа раундов.

Статья Dmitrieva_IMS_2014_rus:
Стандарт открытости представляет собой комплексный документ, состоящий из утвержденной Правительством Российской Федерации Концепции открытости федеральных органов исполнительной власти, а также Методических рекомендаций по реализации принципов открытости в федеральных органах исполнительной власти и Методики мониторинга и оценки открытости федеральных органов исполнительной власти, одобренных Правительственной комиссией по координации деятельности открытого правительства2. Поэтому в систему оценки введены показатели второй стадии, которые в большей степени отражают развитие механизмов участия граждан в деятельно

In [11]:
# ============ T5 ============

print("\nЗагружаем модель T5...")
model_name = "IlyaGusev/rut5_base_sum_gazeta"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

def summarize_with_t5(text):
    input_ids = tokenizer(
        [text],
        max_length=600,
        add_special_tokens=True,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )["input_ids"].to(device)
    output_ids = model.generate(
        input_ids=input_ids,
        min_length=30,
        max_length=250,
        no_repeat_ngram_size=3
    )[0]
    return tokenizer.decode(output_ids, skip_special_tokens=True)

print("\n--- АННОТАЦИИ С T5 ---\n")
t5_annotations = []
for i, row in df.iterrows():
    summary = summarize_with_t5(row['text'])
    t5_annotations.append(summary)
    print(f"Статья {row['name']}:\n{summary}\n")


Загружаем модель T5...

--- АННОТАЦИИ С T5 ---

Статья Adrova_IMS_2013_rus:
Для решения проблемы хранения пароля пользователя в Web-приложениях необходимо использовать соль, которая позволит злоумышленникам получить доступ к 90% паролей социальной сети LinkedIn. В настоящее время даже длинные пароли не могут считаться безопасными, однако для их восстановления можно использовать радужные таблицы, содержащие миллиарды пар «пароль – результат хеширования».

Статья Dmitrieva_IMS_2014_rus:
Система оценки открытости исполнительной власти невозможна без активного участия общества в создании новой модели открытой и прозрачной государственной власти, считают эксперты «Высшей школы экономики».

Статья Khokhlova_IMS_2016_rus:
Большие русскоязычные корпуса текстов М.В. Хохлова создаются автоматически на основе текстов, полученных из Интернета, а затем загружаются в корпус. В настоящее время существует ряд проектов, развивающихся в данном направлении.

Статья MitrofanovaAdamova_IMS_2024_rus:
В Пет

In [22]:
df['annotation_sumy'] = sumy_annotations
df['annotation_t5'] = t5_annotations

# Вывод объединённого DataFrame
print("\n=== ИТОГОВАЯ ТАБЛИЦА ===\n")
print(df[['name', 'annotation_sumy', 'annotation_t5']])

# ✅ Сохранение в CSV с правильной кодировкой
df.to_csv('annotations_combined.csv', index=False, encoding='utf-8-sig')
print("\nФайл сохранён")



=== ИТОГОВАЯ ТАБЛИЦА ===

                              name  \
0              Adrova_IMS_2013_rus   
1           Dmitrieva_IMS_2014_rus   
2           Khokhlova_IMS_2016_rus   
3  MitrofanovaAdamova_IMS_2024_rus   
4               Rogov_IMS_2020_rus   
5               Salin_IMS_2015_rus   
6              Sedova_IMS_2017_rus   
7           Shukshina_IMS_2018_rus   
8                   Tatur_IMS_2023   
9               Zhang_IMS_2019_rus   

                                     annotation_sumy  \
0  Время хеширования этого алгоритма достигает 0,...   
1  Стандарт открытости представляет собой комплек...   
2  Самым известным и популярным корпусом первого ...   
3  В  данной  статье  рассматриваются  следующие ...   
4  Вопрос, связанный с выделением сильных позиций...   
5  При этом чаще всего веб-документы анализируютс...   
6  образуют словосочетание и слово и В процессе р...   
7  Целью работы является представление способа кл...   
8  В сфере аптек замечено повышенное использовани.